# DriverGuard Lite: Tahap 1–3
Dataset: Driver Monitoring System (Roboflow Universe, CC BY 4.0)

**Sebelum mulai:** Runtime → Change runtime type → **T4 GPU**. Simpan API key Roboflow di menu 🔑 *Secrets* (kiri) dengan nama `ROBOFLOW_API_KEY` dan aktifkan *Notebook access*. Jangan tulis API key langsung di notebook.

## 0. Setup

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow

In [ ]:
# Simpan hasil training ke Google Drive supaya tidak hilang kalau Colab terputus
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/driverguard'

## 1. Download dataset
Cek nomor versi terbaru di tab **Dataset** di halaman Roboflow, lalu sesuaikan `VERSION`.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

VERSION = 11  # 8 kelas, 33.069 gambar
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace('driver-monitoring-system-woxvc').project('driver-monitoring-system-v0mei')
dataset = project.version(VERSION).download('yolov11')
DATA = f'{dataset.location}/data.yaml'
print(DATA)
!cat {DATA}

## 2. Cek dataset
Tujuannya: tahu jumlah gambar per split, jumlah box per kelas (apakah seimbang?), dan bentuk box-nya seperti apa.

In [ ]:
import os, glob, yaml, collections
import matplotlib.pyplot as plt

names = yaml.safe_load(open(DATA))['names']
if isinstance(names, dict): names = [names[k] for k in sorted(names)]

counts = {}
for split in ['train', 'valid', 'test']:
    lbls = glob.glob(f'{dataset.location}/{split}/labels/*.txt')
    c = collections.Counter()
    for f in lbls:
        for line in open(f):
            if line.strip(): c[names[int(line.split()[0])]] += 1
    counts[split] = c
    print(f'{split:6s}: {len(lbls):6d} gambar | ' + ', '.join(f'{k}={v}' for k, v in sorted(c.items())))

plt.figure(figsize=(10, 4))
plt.bar(names, [counts['train'][n] for n in names])
plt.title('Jumlah box per kelas (train)'); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

In [ ]:
# Lihat 12 sampel acak beserta box-nya
import random, cv2
imgs = random.sample(glob.glob(f'{dataset.location}/train/images/*'), 12)
fig, axes = plt.subplots(3, 4, figsize=(16, 11))
for ax, p in zip(axes.flat, imgs):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    lp = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    for line in open(lp) if os.path.exists(lp) else []:
        c, x, y, bw, bh = map(float, line.split()[:5])
        x1, y1, x2, y2 = int((x-bw/2)*w), int((y-bh/2)*h), int((x+bw/2)*w), int((y+bh/2)*h)
        cv2.rectangle(im, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(im, names[int(c)], (x1, max(15, y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    ax.imshow(im); ax.axis('off')
plt.tight_layout(); plt.show()

**Catat di README:** kelas mana yang jumlahnya sedikit, dan apakah ada label yang terlihat salah. Ini bahan *error analysis* nanti.

## 3. Training
Mulai dari model *nano* (paling ringan, cocok untuk CPU laptop). `patience` menghentikan training otomatis kalau validasi tidak membaik.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')  # pretrained COCO -> transfer learning
model.train(data=DATA, epochs=30, imgsz=640, batch=16, patience=10,
            project=SAVE_DIR, name='v11_run1', exist_ok=True)

**Cadangan darurat.** Kalau Colab terputus: jalankan sel Setup, mount Drive, dan download dataset, lalu hapus tanda `#` di bawah dan jalankan. Jangan jalankan sel training di atas lagi.

In [ ]:
# model = YOLO(f'{SAVE_DIR}/v11_run1/weights/last.pt')
# model.train(resume=True)

## 4. Evaluasi di test set

In [ ]:
best = YOLO(f'{SAVE_DIR}/v11_run1/weights/best.pt')
metrics = best.val(data=DATA, split='test')
print('mAP@0.5     :', round(metrics.box.map50, 3))
print('mAP@0.5:0.95:', round(metrics.box.map, 3))
for i, n in enumerate(names):
    print(f'{n:12s} AP50 = {metrics.box.ap50[i]:.3f}')

File `best.pt` ada di Google Drive: `MyDrive/driverguard/v11_run1/weights/best.pt`. Unduh file itu ke laptop untuk tahap 4 (program webcam).